
# Week 6: Robustness Checks

**Owner:** Fatima Nasir  
**Role:** Evaluation / Backtesting Lead  
**Deliverable:** Robustness Check Results

## Objective

Evaluate the consistency of market stress detection results using different volatility window lengths and prediction horizons.

## Tasks

1. Re-run the evaluation using different window lengths and prediction horizons.
2. Compare the VIX-only baseline with the combined market indicators.
3. Analyze changes in model performance across different settings.
4. Summarize the robustness check results and limitations.


In [10]:

# Step 1: Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("Libraries loaded successfully!")


Libraries loaded successfully!



## 1. Data Preparation

Load the historical market indicators and stress labels required for robustness testing. The data will be used to compare model performance across different volatility windows and prediction horizons.


In [11]:

# Step 2: Load market data

ROOT = Path.cwd().parent

market = pd.read_parquet(
    ROOT / "data" / "processed" / "daily_panel.parquet"
)

print("Dataset shape:", market.shape)

print("\nDate range:")
print(market.index.min(), "to", market.index.max())

print("\nAvailable columns:")
print(market.columns.tolist())

display(market.head(3))


Dataset shape: (6997, 28)

Date range:
1999-01-05 00:00:00 to 2026-10-28 00:00:00

Available columns:
['SPY', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY', 'SPY_ret', 'XLB_ret', 'XLE_ret', 'XLF_ret', 'XLI_ret', 'XLK_ret', 'XLP_ret', 'XLU_ret', 'XLV_ret', 'XLY_ret', 'STLFSI4', 'NFCI', 'VIXCLS', 'BAMLH0A0HYM2', 'T10Y2Y', 'stress', 'avg_corr_60d', 'absorption_250d']


,SPY,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,...,XLV_ret,XLY_ret,STLFSI4,NFCI,VIXCLS,BAMLH0A0HYM2,T10Y2Y,stress,avg_corr_60d,absorption_250d
date,,,,,,,,,,,,,,,,,,,,,
1999-01-05,76.952858,6.040749,5.646690,11.248712,15.244627,12.562191,14.132486,5.807241,17.525572,9.684027,...,0.015541,0.016707,0.3872,-0.201,24.46,NaN,0.10,0,NaN,NaN
1999-01-06,78.808105,6.147099,5.833012,11.605337,15.572269,12.933377,14.303343,5.867515,17.816624,9.959071,...,0.016471,0.028006,0.3872,-0.201,23.34,NaN,0.11,0,NaN,NaN
1999-01-07,78.421593,6.100304,5.806396,11.783652,15.437354,12.892775,14.140624,5.846416,17.733469,9.947616,...,-0.004678,-0.001151,0.3872,-0.201,24.37,NaN,0.14,0,NaN,NaN



## 2. Volatility Window Robustness

Realized volatility is calculated using 10-day, 21-day, and 42-day rolling windows. Comparing these windows helps assess how sensitive the identification of market stress is to the selected window length.


In [12]:

# Step 3: Calculate volatility for different windows

windows = [10, 21, 42]

spy_returns = market["SPY"].pct_change(fill_method=None)

volatility = pd.DataFrame(index=market.index)

for window in windows:
    volatility[f"vol_{window}d"] = (
        spy_returns.rolling(window, center=True).std()
        * np.sqrt(252)
    )

print("Volatility dataset shape:", volatility.shape)

print("\nMissing values:")
print(volatility.isna().sum())

display(volatility.dropna().head())


Volatility dataset shape: (6997, 3)

Missing values:
vol_10d    10
vol_21d    21
vol_42d    42
dtype: int64


,vol_10d,vol_21d,vol_42d
date,,,
1999-02-05,0.214965,0.207598,0.215021
1999-02-08,0.215045,0.207236,0.207041
1999-02-09,0.215115,0.215575,0.207681
1999-02-10,0.214644,0.208397,0.208640
1999-02-11,0.211139,0.204985,0.208648



### Stress Classification Across Different Windows

A 25% annualized volatility threshold is applied to each rolling window. The resulting stress classifications are compared to examine how the window length affects the number of identified stress days.


In [13]:

# Step 4: Compare stress classifications

threshold = 0.25

stress_windows = pd.DataFrame(index=volatility.index)

for window in windows:
    vol_col = f"vol_{window}d"
    stress_col = f"stress_{window}d"

    stress_windows[stress_col] = (
        volatility[vol_col] >= threshold
    ).astype(float)

    # Keep missing volatility values as missing labels
    stress_windows.loc[
        volatility[vol_col].isna(), stress_col
    ] = np.nan

summary = pd.DataFrame({
    "Window": windows,
    "Stress Days": [
        int(stress_windows[f"stress_{w}d"].sum())
        for w in windows
    ],
    "Stress Percentage": [
        round(stress_windows[f"stress_{w}d"].mean() * 100, 2)
        for w in windows
    ]
})

display(summary)


,Window,Stress Days,Stress Percentage
0,10,875,12.52
1,21,882,12.64
2,42,871,12.52



### Stress Classification Agreement

The 21-day volatility window is used as the reference for comparing stress classifications. Agreement rates are calculated to examine whether shorter and longer windows identify the same stress days.


In [14]:

# Step 5: Compare stress classification agreement

agreement_results = []

for window in [10, 42]:
    comparison_df = stress_windows[
        ["stress_21d", f"stress_{window}d"]
    ].dropna()

    agreement = (
        comparison_df["stress_21d"]
        == comparison_df[f"stress_{window}d"]
    ).mean() * 100

    different_days = (
        comparison_df["stress_21d"]
        != comparison_df[f"stress_{window}d"]
    ).sum()

    agreement_results.append({
        "Comparison": f"21-day vs {window}-day",
        "Agreement (%)": round(agreement, 2),
        "Different Days": int(different_days)
    })

display(pd.DataFrame(agreement_results))


,Comparison,Agreement (%),Different Days
0,21-day vs 10-day,95.00,349
1,21-day vs 42-day,95.15,337



## 3. Model Robustness Across Volatility Windows

The VIX-only baseline and combined market indicators are evaluated using stress classifications from different volatility windows.

A chronological train/test split is used to compare model performance while keeping the evaluation consistent across window lengths.


In [15]:

# Step 6: Prepare data for model robustness testing

features = [
    "VIXCLS",
    "STLFSI4",
    "NFCI",
    "SPY_ret",
    "avg_corr_60d",
    "absorption_250d"
]

robustness_df = market[features].join(
    stress_windows,
    how="inner"
)

# Keep dates with complete indicators and labels
robustness_df = robustness_df.dropna().sort_index()

print("Dataset shape:", robustness_df.shape)

print("\nDate range:")
print(robustness_df.index.min(), "to",
      robustness_df.index.max())

print("\nStress label distribution:")
for window in windows:
    print(f"\n{window}-day window:")
    print(robustness_df[f"stress_{window}d"].value_counts())


Dataset shape: (6728, 9)

Date range:
1999-12-30 00:00:00 to 2026-09-30 00:00:00

Stress label distribution:

10-day window:
stress_10d
0.0    5866
1.0     862
Name: count, dtype: int64

21-day window:
stress_21d
0.0    5850
1.0     878
Name: count, dtype: int64

42-day window:
stress_42d
0.0    5857
1.0     871
Name: count, dtype: int64



### VIX Baseline and Combined Model Comparison

Logistic regression models are evaluated using stress labels from the 10-day, 21-day, and 42-day volatility windows. A VIX-only baseline is compared with a model using additional market indicators.

The same chronological training and testing periods are used across all windows to assess the consistency of model performance.


In [16]:

# Step 7: Compare models across volatility windows

split_idx = int(len(robustness_df) * 0.80)

train_df = robustness_df.iloc[:split_idx]
test_df = robustness_df.iloc[split_idx:]

results = []

for window in windows:
    target = f"stress_{window}d"

    y_train = train_df[target].astype(int)
    y_test = test_df[target].astype(int)

    for model_name, cols in [
        ("VIX Only", ["VIXCLS"]),
        ("Combined", features)
    ]:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )

        model.fit(train_df[cols], y_train)

        predictions = model.predict(test_df[cols])
        probabilities = model.predict_proba(
            test_df[cols]
        )[:, 1]

        tn, fp, fn, tp = confusion_matrix(
            y_test, predictions, labels=[0, 1]
        ).ravel()

        results.append({
            "Window": window,
            "Model": model_name,
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1": f1_score(
                y_test, predictions, zero_division=0
            ),
            "ROC-AUC": roc_auc_score(
                y_test, probabilities
            ),
            "PR-AUC": average_precision_score(
                y_test, probabilities
            ),
            "False Alarm Rate": fp / (fp + tn)
        })

window_results = pd.DataFrame(results)

display(window_results.round(4))


,Window,Model,Precision,Recall,F1,ROC-AUC,PR-AUC,False Alarm Rate
0,10,VIX Only,0.4619,0.8509,0.5988,0.9671,0.7660,0.0917
1,10,Combined,0.4929,0.9123,0.6400,0.9750,0.8133,0.0869
2,21,VIX Only,0.5024,0.8125,0.6209,0.9451,0.6953,0.0846
3,21,Combined,0.5000,0.8906,0.6404,0.9615,0.7590,0.0936
4,42,VIX Only,0.5441,0.7551,0.6325,0.9354,0.6213,0.0776
5,42,Combined,0.5484,0.8095,0.6538,0.9499,0.6717,0.0817



## 4. Prediction Horizon Robustness

Different prediction horizons are examined to evaluate how the stress classification changes when the target date moves further into the future.

The analysis uses 1-day, 5-day, and 10-day horizons. These tests are based on historical stress labels and are interpreted as retrospective robustness checks.


In [17]:

# Step 9: Prepare different prediction horizons

horizons = [1, 5, 10]

horizon_df = robustness_df.copy()

for horizon in horizons:
    horizon_df[f"stress_ahead_{horizon}d"] = (
        horizon_df["stress_21d"].shift(-horizon)
    )

print("Stress labels at different horizons:")

for horizon in horizons:
    col = f"stress_ahead_{horizon}d"
    print(f"\n{horizon}-day horizon:")
    print(horizon_df[col].value_counts(dropna=False))


Stress labels at different horizons:

1-day horizon:
stress_ahead_1d
0.0    5850
1.0     877
NaN       1
Name: count, dtype: int64

5-day horizon:
stress_ahead_5d
0.0    5850
1.0     873
NaN       5
Name: count, dtype: int64

10-day horizon:
stress_ahead_10d
0.0    5850
1.0     868
NaN      10
Name: count, dtype: int64



### Model Comparison Across Prediction Horizons

The VIX-only and combined models are evaluated using 1-day, 5-day, and 10-day target horizons.

Model performance is compared using recall, F1-score, ROC-AUC, PR-AUC, and false alarm rate to examine how the results change as the prediction horizon increases.


In [18]:

# Step 10: Compare models across prediction horizons

horizon_results = []

for horizon in horizons:
    target = f"stress_ahead_{horizon}d"

    # Remove missing target values
    data = horizon_df.dropna(subset=[target]).copy()

    split_idx = int(len(data) * 0.80)
    train = data.iloc[:split_idx]
    test = data.iloc[split_idx:]

    y_train = train[target].astype(int)
    y_test = test[target].astype(int)

    for model_name, cols in [
        ("VIX Only", ["VIXCLS"]),
        ("Combined", features)
    ]:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )

        model.fit(train[cols], y_train)

        predictions = model.predict(test[cols])
        probabilities = model.predict_proba(
            test[cols]
        )[:, 1]

        tn, fp, fn, tp = confusion_matrix(
            y_test, predictions, labels=[0, 1]
        ).ravel()

        horizon_results.append({
            "Horizon": horizon,
            "Model": model_name,
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1": f1_score(
                y_test, predictions, zero_division=0
            ),
            "ROC-AUC": roc_auc_score(
                y_test, probabilities
            ),
            "PR-AUC": average_precision_score(
                y_test, probabilities
            ),
            "False Alarm Rate": (
                fp / (fp + tn) if (fp + tn) else np.nan
            )
        })

horizon_results = pd.DataFrame(horizon_results)

display(horizon_results.round(4))


,Horizon,Model,Precision,Recall,F1,ROC-AUC,PR-AUC,False Alarm Rate
0,1,VIX Only,0.4905,0.8047,0.6095,0.9430,0.6826,0.0878
1,1,Combined,0.4914,0.8906,0.6333,0.9587,0.7382,0.0969
2,5,VIX Only,0.4749,0.8125,0.5994,0.9343,0.6025,0.0945
3,5,Combined,0.4270,0.8906,0.5772,0.9479,0.6297,0.1257
4,10,VIX Only,0.4202,0.7812,0.5464,0.9167,0.4849,0.1135
5,10,Combined,0.3314,0.8984,0.4842,0.9262,0.5018,0.1908



### Prediction Horizon Robustness Results

The combined model achieved higher recall and PR-AUC than the VIX-only baseline across the 1-day, 5-day, and 10-day horizons.

Recall remained relatively stable for the combined model, while PR-AUC decreased as the prediction horizon increased.

The combined model achieved a higher F1-score at the 1-day horizon, but the VIX-only model performed better on F1-score at the 5-day and 10-day horizons.

False alarm rates also increased at longer horizons, particularly for the combined model.

Overall, the additional market indicators improved stress detection in terms of recall and PR-AUC, but the results show a trade-off between identifying stress periods and generating false alarms.

These tests use shifted historical labels based on centered volatility and do not establish real-time forecasting performance.



## 5. Conclusion and Limitations

### Key Findings

The robustness checks evaluated market stress detection using three volatility windows (10, 21, and 42 days) and three target horizons (1, 5, and 10 days).

Across all volatility windows, the combined model achieved higher recall, F1-score, and PR-AUC than the VIX-only baseline.

Across the prediction horizons, the combined model consistently improved recall and PR-AUC. However, the F1-score was lower at the 5-day and 10-day horizons, and false alarm rates were generally higher.

The results suggest that additional market indicators provide useful information beyond VIX, although performance depends on the evaluation settings.

### Limitations

- Stress labels are based on centered rolling volatility and include future market observations.
- SPY returns were used as a proxy for the S&P 500 index in the window robustness tests.
- The window tests use a simple volatility threshold rather than the original episode-adjusted stress labels.
- The horizon tests shift historical labels and do not demonstrate real-time early-warning performance.
- Further testing with real-time available features and forward-looking targets is needed.

### Next Steps

Future work should evaluate additional market periods, use strictly forward-looking stress targets, and examine whether model improvements remain consistent under different market conditions.
